# How Much Did It Rain? (starter)

Read `README.md` first. This notebook loads the data, trains a plain baseline on the daily columns, and scores it. Your job: beat it.

In [1]:
import numpy as np, pandas as pd

TARGET = "rain_sum (mm)"
train = pd.read_csv("data/train.csv", parse_dates=["time"])
test = pd.read_csv("data/test.csv", parse_dates=["time"])
train_hourly = pd.read_csv("data/train_hourly.csv", parse_dates=["time"])
test_hourly = pd.read_csv("data/test_hourly.csv", parse_dates=["time"])
print(train.shape, test.shape, train_hourly.shape, test_hourly.shape)
train.head()

(7900, 16) (5180, 16) (189600, 32) (124320, 32)


,time,temperature_2m_max (°C),temperature_2m_min (°C),apparent_temperature_max (°C),apparent_temperature_min (°C),sunrise (iso8601),sunset (iso8601),shortwave_radiation_sum (MJ/m²),rain_sum (mm),snowfall_sum (cm),windspeed_10m_max (km/h),windgusts_10m_max (km/h),winddirection_10m_dominant (°),et0_fao_evapotranspiration (mm),elevation,city
0,2018-01-01,29.9,26.0,36.3,31.6,2018-01-01T05:15,2018-01-01T17:49,17.59,7.7,0.0,6.9,20.2,277.0,3.61,0,su
1,2018-01-02,30.6,25.7,37.7,31.2,2018-01-02T05:15,2018-01-02T17:50,19.87,9.4,0.0,7.1,18.0,168.0,4.11,0,su
2,2018-01-03,31.8,25.9,40.3,31.7,2018-01-03T05:16,2018-01-03T17:50,20.44,7.8,0.0,8.1,21.2,125.0,4.23,0,su
3,2018-01-04,30.8,26.0,39.1,31.6,2018-01-04T05:16,2018-01-04T17:50,20.44,29.7,0.0,7.2,21.6,190.0,4.22,0,su
4,2018-01-05,30.9,25.2,37.6,29.9,2018-01-05T05:17,2018-01-05T17:51,20.10,22.5,0.0,6.9,21.2,241.0,4.16,0,su


## Scoring helper
Use it only at the end. For decisions, validate on the last months of train.

In [2]:
def score(pred):
    """pred: array of rain predictions, same order as test.csv"""
    labels = pd.read_csv("data/test_labels.csv").set_index("id").loc[test["id"], TARGET].values
    rmse = np.sqrt(np.mean((np.clip(pred, 0, None) - labels) ** 2))
    print(f"test RMSE = {rmse:.4f}")
    return rmse

## Baseline: XGBoost on the daily numeric columns

In [3]:
from xgboost import XGBRegressor

train = train[train[TARGET].notna()]
features = [c for c in test.columns if c not in ("id", "time", "city", "sunrise (iso8601)", "sunset (iso8601)")]

model = XGBRegressor(n_estimators=600, learning_rate=0.03, max_depth=5, random_state=42)
model.fit(train[features], train[TARGET])
pred = np.clip(model.predict(test[features]), 0, None)

## Your predictions
One row per test day, columns `id, rain_sum (mm)`.

In [4]:
submission = pd.DataFrame({"id": test["id"], TARGET: pred})
submission.head()

,id,rain_sum (mm)
0,0,7.331112
1,1,5.038433
2,2,0.473239
3,3,2.445124
4,4,12.299035


In [5]:
score(submission[TARGET])

test RMSE = 6.3695


np.float64(6.369498837042483)

## Ideas to try
- The hourly files have 24 rows per day. How can you turn them into daily columns?
- When it rains, what happens to the soil?
- Which city is which kind of climate? Does the season matter?
- Build a validation set from the **last months of train** before touching the test score.